# House Prices Competition - Modeling Handover

**Last Updated:** 2026-05-25
**Project:** Kaggle House Prices - Log-transformed SalePrice prediction
**Status:** Initial modeling complete, validation needed on key components

---

## 1. Quick Reference

| Item | Value |
|------|-------|
| **Target** | log(SalePrice) - use `np.expm1()` to convert back |
| **Metric** | RMSE on log scale → RMSLE on original |
| **CV Strategy** | 5-fold KFold (shuffled, random_state=42) |
| **Current Best** | Ensemble (Ridge + XGBoost + LightGBM) |
| **Ensemble Method** | Inverse RMSE weighting (from CV scores) |

### Current CV Performance (log-scale RMSE)
| Model | Mean RMSE | Std |
|-------|-----------|-----|
| Ridge (alpha=?) | *to be filled from run* | *to be filled* |
| XGBoost (optimized) | *to be filled* | *to be filled* |
| LightGBM (optimized) | *to be filled* | *to be filled* |
| Ensemble | *to be filled* | - |

---

## 2. ⚠️ CRITICAL: Suspect Areas (Needs Validation)

### 2.1 Target Encoding - Potential Leakage

**Location:** `apply_target_encoding()` function

**The Concern:**
```python
def apply_target_encoding(X_train_fold, y_train_fold, X_val_fold, columns, global_mean, smoothing=20):
    # Uses global_mean (mean of ALL training data) in smoothing calculation
    smoothed = (group_means['mean'] * group_means['count'] + global_mean * smoothing) / ...
```

**Why this might leak:**
- `global_mean` is computed from the ENTIRE training set before CV
- Each fold's encoding uses information from ALL folds (via global_mean)
- This is subtle but could inflate CV performance

**What to test:**
```python
# Quick test - remove global_mean dependency
def apply_target_encoding_no_leakage(X_train_fold, y_train_fold, X_val_fold, columns, smoothing=20):
    # Use training fold's own mean instead
    fold_mean = y_train_fold.mean()
    # ... rest of calculation using fold_mean
```

**Other encoding issues:**
- We drop original categorical columns after encoding - is this correct? (Some models benefit from both)
- Smoothing factor (20) is hardcoded - should be tuned or removed
- Unknown categories in test set get global_mean (might be fine, but test)

### 2.2 Ensemble Weighting - Too Simplistic?

**Current approach:**
```python
weights = {
    'ridge': 1 / np.mean(ridge_scores),
    'xgb': 1 / np.mean(xgb_scores),
    'lgb': 1 / np.mean(lgb_scores)
}
# Normalized to sum to 1
```

**Problems:**
1. Weights derived from CV but applied to full model (different training set)
2. Simple inverse RMSE doesn't account for correlation between models
3. No validation that weights generalize to test set

**Better alternatives to explore:**
- Stacking with a meta-model (Ridge or simple linear model)
- Rank-based weighting (weight by CV rank, not raw RMSE)
- Optimize weights on validation set (different from training)

### 2.3 Model Training Concerns

**Overfitting risk:**
- Optuna runs on same folds used for final evaluation (3-fold tuning → 5-fold evaluation)
- No separate holdout set for final verification
- Feature count includes target encoding columns (~original features + 4 encoded)

**What's missing:**
- Learning curves to diagnose bias/variance
- Out-of-fold prediction correlations between models
- Test-time performance tracking (can't do until Kaggle submission)

---

## 3. Data Pipeline

### Input Files (must exist)
```
../data/X_train_processed.csv      # Training features
../data/y_train_processed.csv      # Log-transformed SalePrice
../data/X_test_processed.csv       # Test features
../data/test.csv                   # Original test (for IDs only)
```

### Critical Columns for Target Encoding
```python
target_encode_cols = ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
```
*These MUST exist in processed data - check if preprocessing dropped any*

### Output Files Generated
```
../submissions/submission_YYYYMMDD_HHMMSS.csv    # Kaggle submission
../models/model_artifacts.pkl                   # All models + metadata
```

### Column Alignment Gotcha
**Location:** Section 11 - Test prediction generation

**What can break:**
- If training and test have different columns after encoding
- Our fix adds missing columns with 0 and drops extras
- But this could mask real data issues (e.g., category in test not seen in training)

**Debug command:**
```python
# After alignment, check for all-zero columns in test
zero_cols = (X_test_final == 0).all()
print(f"Columns that are all zeros in test: {zero_cols[zero_cols].index.tolist()}")
```

---

## 4. Current Hyperparameters

### Ridge (from tuning section)
```python
best_alpha = None  # FILL IN FROM ACTUAL RUN
# Range tested: [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]
```

### XGBoost (from Optuna - 30 trials, 3-fold CV)
```python
xgb_best_params = {
    'n_estimators': None,  # FILL IN (300-1200 range)
    'max_depth': None,      # FILL IN (3-10 range)
    'learning_rate': None,  # FILL IN (0.01-0.15)
    'subsample': None,      # FILL IN (0.6-1.0)
    'colsample_bytree': None, # FILL IN (0.6-1.0)
    'reg_alpha': None,      # FILL IN (1e-4 to 5)
    'reg_lambda': None,     # FILL IN (1e-4 to 5)
}
```

### LightGBM (from Optuna - 30 trials, 3-fold CV)
```python
lgb_best_params = {
    'n_estimators': None,   # FILL IN (300-1200 range)
    'num_leaves': None,     # FILL IN (15-100 range)
    'max_depth': None,      # FILL IN (3-12 range)
    'learning_rate': None,  # FILL IN (0.01-0.15)
    'subsample': None,      # FILL IN (0.6-1.0)
    'colsample_bytree': None, # FILL IN (0.6-1.0)
    'reg_alpha': None,      # FILL IN (1e-4 to 5)
    'reg_lambda': None,     # FILL IN (1e-4 to 5)
}
```

---

## 5. Performance & Debugging Tools

### 5.1 Validate Target Encoding (No Leakage Check)

Add this function to test encoding isolation:

```python
def test_target_encoding_no_leakage():
    """Verify that fold encodings don't use global statistics"""
    from sklearn.model_selection import KFold
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    global_mean = y_train.mean()
    
    for train_idx, val_idx in kf.split(X_train):
        X_train_fold = X_train.iloc[train_idx]
        y_train_fold = y_train[train_idx]
        X_val_fold = X_train.iloc[val_idx]
        
        # Compute fold-specific mean
        fold_mean = y_train_fold.mean()
        
        # Compare with global_mean
        print(f"Fold mean: {fold_mean:.4f}, Global mean: {global_mean:.4f}")
        print(f"Difference: {abs(fold_mean - global_mean):.4f}")
        
        # If difference > 0.01, global_mean is influencing encoding
        # (depends on target distribution)
```

### 5.2 Quick vs Full Validation

**Quick validation** (for debugging, ~2-3 minutes):
```python
# Use 3 folds instead of 5
# Reduce Optuna trials to 10-15
# Use subset of features (optional)
```

**Full validation** (for final results, ~30-60 minutes):
```python
# Current settings: 5 folds, 30 Optuna trials per model
# This is what the code does by default
```

### 5.3 CPU Fallback (No GPU)

Add this to the top of modeling cells:

```python
import subprocess
import sys

def get_device_config():
    """Auto-detect GPU availability for XGBoost/LightGBM"""
    try:
        # Check for CUDA
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            return {'xgb_device': 'cuda', 'lgb_device': 'gpu'}
    except FileNotFoundError:
        pass
    
    print("⚠️ No GPU detected - using CPU (will be slower)")
    return {'xgb_device': 'cpu', 'lgb_device': 'cpu'}

devices = get_device_config()
# Then use devices['xgb_device'] and devices['lgb_device'] in model params
```

---

## 6. Extension Points

### 6.1 Visualizations You Requested

**Predicted vs Actual (with residuals):**
```python
def plot_predictions(y_true, y_pred, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Scatter plot
    axes[0].scatter(y_true, y_pred, alpha=0.5)
    axes[0].plot([y_true.min(), y_true.max()], 
                 [y_true.min(), y_true.max()], 'r--', lw=2)
    axes[0].set_xlabel('Actual log(Price)')
    axes[0].set_ylabel('Predicted log(Price)')
    axes[0].set_title(f'{model_name}: Predicted vs Actual')
    
    # Residuals
    residuals = y_true - y_pred
    axes[1].hist(residuals, bins=50, edgecolor='black')
    axes[1].axvline(x=0, color='r', linestyle='--')
    axes[1].set_xlabel('Residuals')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title(f'Residual Distribution (RMSE: {rmse(y_true, y_pred):.4f})')
    
    plt.tight_layout()
    plt.show()
```

**Feature Impact Plots:**
```python
def plot_feature_impact(model, X, feature_name, model_name):
    """Partial dependence-style plot for a single feature"""
    X_temp = X.copy()
    feature_values = np.percentile(X[feature_name], np.linspace(0, 100, 50))
    predictions = []
    
    for val in feature_values:
        X_temp[feature_name] = val
        predictions.append(model.predict(X_temp).mean())
    
    plt.plot(feature_values, predictions)
    plt.xlabel(feature_name)
    plt.ylabel('Predicted log(Price)')
    plt.title(f'{model_name}: Impact of {feature_name}')
    plt.show()
```

### 6.2 Regularization Exploration (You Requested)

**For Ridge:**
```python
# Extended alpha search
alphas = np.logspace(-3, 2, 50)  # 0.001 to 100
# Current only tested 10 values
```

**For XGBoost/LightGBM:**
```python
# Focus tuning on regularization parameters only
def objective_regularization_only(trial):
    params = {
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        # Keep other params fixed at best values from initial run
    }
    # ... rest of objective
```

### 6.3 Better Ensemble Methods

**Stacking (recommended over weighting):**
```python
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

stacking_model = StackingRegressor(
    estimators=[
        ('ridge', ridge_full),
        ('xgb', xgb_full),
        ('lgb', lgb_full)
    ],
    final_estimator=Ridge(alpha=1.0),  # Meta-model
    cv=5  # Use CV to prevent overfitting
)
```

---

## 7. Optimization Opportunities

### Speed Improvements
| Issue | Current | Suggested |
|-------|---------|-----------|
| Optuna CV | 3 full folds inside objective | Pre-compute fold indices, use early stopping |
| Target encoding | Recomputes group stats each fold | Cache category statistics |
| Model training | Full retraining for each trial | Use warm start if possible |

### Memory Usage
- Current: Keeps original + encoded columns simultaneously
- Fix: Drop original columns immediately after encoding

### Stability
- Add random_state to ALL model instances (some missing)
- Set `deterministic=True` for XGBoost (if using CUDA)
- Save intermediate results (checkpointing)

---

## 8. Immediate To-Do List

### 🔴 High Priority (Before Next Run)
- [ ] **Validate target encoding for leakage** (use test function in 5.1)
- [ ] **Add CPU fallback logic** (code in 5.3)
- [ ] **Fill in current hyperparameters** (section 4 - after running)
- [ ] **Test column alignment** (run debug command from section 3)

### 🟡 Medium Priority (Next Sprint)
- [ ] **Add prediction vs actual plots** (code in 6.1)
- [ ] **Try stacking ensemble** (code in 6.3) - compare with weighting
- [ ] **Extended regularization search for Ridge** (50+ alphas)
- [ ] **Add learning curves** (diagnose overfitting)

### 🟢 Low Priority (Future)
- [ ] **Feature importance visualization** for all models
- [ ] **CatBoost as additional model** (often good for categorical data)
- [ ] **Hyperparameter tuning with 5-fold CV** (currently 3-fold for speed)
- [ ] **Automated test for data drift** (train vs test column distributions)

---

## 9. Common Debugging Commands

```python
# Run a single fold only (fast debugging)
single_fold_indices = [list(kf.split(X_train))[0]]
# Modify evaluate_with_target_encoding to accept fold_indices parameter

# Check if any feature has zero variance
zero_var_cols = X_train.columns[X_train.std() == 0]
print(f"Zero variance columns: {zero_var_cols.tolist()}")

# Verify no infinite values
print(f"Infinite values in train: {np.isinf(X_train).sum().sum()}")
print(f"Infinite values in test: {np.isinf(X_test).sum().sum()}")

# Check target encoding mapping consistency
for col in target_encode_cols:
    train_cats = set(X_train[col].unique())
    test_cats = set(X_test[col].unique())
    unseen = test_cats - train_cats
    if unseen:
        print(f"Warning: {col} has unseen categories in test: {unseen}")
```

---

## 10. Questions for Future Discussions

1. **Target encoding:** Should we switch to leave-one-out encoding or CatBoost's native categorical handling?
2. **Validation strategy:** Do we need a separate holdout set (e.g., 20% of training) for final verification?
3. **Feature engineering:** Should we add interaction features between top encoded columns?
4. **Deployment:** Are we just generating a submission, or will this need to be productionized?
5. **Hyperparameter tuning:** Increase Optuna trials from 30 to 100+ for final run?

---

## Appendix: Code Structure Reference

| Section | Purpose | Key Functions/Variables |
|---------|---------|------------------------|
| 3 | Target encoding | `apply_target_encoding()` |
| 5 | CV evaluation | `evaluate_with_target_encoding()` |
| 6 | Ridge baseline | `best_alpha`, ridge tuning loop |
| 7 | XGBoost | `objective_xgb()`, `study_xgb` |
| 8 | LightGBM | `objective_lgb()`, `study_lgb` |
| 9 | Ensemble | `weights`, weighted average |
| 10 | Full training | `X_train_final`, full model fitting |
| 11 | Test predictions | Column alignment logic |
| 12-14 | Output | Submission creation, artifacts |

---

**Next Action:** Run the code once to get actual hyperparameter values and CV scores, then update section 1 and 4. Then run target encoding validation (section 5.1) to confirm no leakage.

**Ready for iterative improvements!**
```